# Day 8 — Embeddings & semantic similarity

**Phase 1 · Generative AI Foundations · ~35 min · Needs a provider (optional)**

## 🎯 Objective
Understand **embeddings** — turning text into vectors of *meaning* — and measure how
similar two pieces of text are with **cosine similarity**. This is the engine
behind search and RAG (Phase 7).

## 🧠 Concept
An embedding maps text to a list of numbers (a vector) so that *similar meaning →
nearby vectors*. We compare vectors with **cosine similarity** (1 = identical
direction, 0 = unrelated).

```mermaid
flowchart LR
    A["'a happy dog'"] --> EA["[0.1, 0.9, ...]"]
    B["'a joyful puppy'"] --> EB["[0.12, 0.88, ...]"]
    EA & EB --> C["cosine ≈ 0.97 ✅ similar"]
```

To learn the math with **zero setup**, today's core exercise uses simple
*word-count* vectors. The 🔒 solution also shows the **real** version using the
model's embeddings via `shared.llm.embed` (optional).

## 🔬 Exercise
Implement `cosine(a, b)` and rank sentences by similarity to a query.

## ✅ Done when
- The most semantically related sentence ranks highest.

## 📝 Reflect
1. Why do *count* vectors miss that "dog" and "puppy" are similar — and how do
   learned embeddings fix that?
2. How would you use this to find the right document for a question? (That's RAG.)

## 🔭 Tomorrow
Day 9: the limits — hallucinations, context, and prompt-injection safety.


### ▶ Run this first

In [ ]:
# Run me first: make the course's shared/ package importable from this notebook.
import sys, pathlib
for _cand in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_cand / 'shared' / 'llm.py').exists():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break


## 🔬 Your turn
Complete the `TODO`s below, then run the cell (**Shift+Enter**).

In [ ]:
import math


def vectorize(text, vocab):
    words = text.lower().split()
    return [words.count(w) for w in vocab]


def cosine(a, b):
    """Cosine similarity of two equal-length vectors.

    TODO: dot = sum(x*y for x, y in zip(a, b))
          na = math.sqrt(sum(x*x for x in a)); nb = math.sqrt(sum(y*y for y in b))
          return 0.0 if na == 0 or nb == 0 else dot / (na * nb)
    """
    raise NotImplementedError


SENTENCES = [
    "the cat sat on the mat",
    "a feline rested on the rug",
    "the stock market fell sharply today",
]
QUERY = "the cat sat on the mat"

if __name__ == "__main__":
    vocab = sorted({w for s in SENTENCES + [QUERY] for w in s.lower().split()})
    qv = vectorize(QUERY, vocab)
    ranked = sorted(SENTENCES, key=lambda s: cosine(qv, vectorize(s, vocab)), reverse=True)
    for s in ranked:
        print(f"{cosine(qv, vectorize(s, vocab)):.3f}  {s}")


---
---
## 🔒 Solution
*Try it yourself first!* Run the cell below only when you're ready to compare.

In [ ]:
import math


def vectorize(text, vocab):
    words = text.lower().split()
    return [words.count(w) for w in vocab]


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return 0.0 if na == 0 or nb == 0 else dot / (na * nb)


SENTENCES = [
    "the cat sat on the mat",
    "a feline rested on the rug",
    "the stock market fell sharply today",
]
QUERY = "the cat sat on the mat"


def real_embeddings_demo():
    """Optional: the REAL version using learned embeddings.

    Needs an embedding model (OpenAI/Azure, or Ollama: `ollama pull nomic-embed-text`).
    Notice 'a feline rested on the rug' scores high even with NO shared words.
    """
    from shared.llm import embed
    vecs = embed(SENTENCES + [QUERY])
    qv = vecs[-1]
    for s, v in zip(SENTENCES, vecs[:-1]):
        print(f"{cosine(qv, v):.3f}  {s}")


if __name__ == "__main__":
    print("— word-count vectors —")
    vocab = sorted({w for s in SENTENCES + [QUERY] for w in s.lower().split()})
    qv = vectorize(QUERY, vocab)
    for s in sorted(SENTENCES, key=lambda s: cosine(qv, vectorize(s, vocab)), reverse=True):
        print(f"{cosine(qv, vectorize(s, vocab)):.3f}  {s}")

    # Uncomment to try real embeddings:
    # print("\n— learned embeddings —"); real_embeddings_demo()
